In [ ]:
# !unzip cifar-10-batches-bin.zip
# !unzip cuML.zip
# !unzip content.zip
# !unzip header.zip
# !unzip lib.zip
# !unzip source.zip
!unzip models.zip
# !unzip template.zip

Archive:  models.zip
   creating: models/
  inflating: models/model.dat        


In [ ]:
!cmake -S . -B /Build

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /Build


In [ ]:
!cmake --build Build --config Release

[  2%] Building CUDA object CMakeFiles/train.dir/source/dataset.cpp.o
[  5%] Building CUDA object CMakeFiles/train.dir/train.cpp.o
[  8%] Linking CUDA executable train
[ 33%] Built target train
[ 36%] Building CUDA object CMakeFiles/encode.dir/source/dataset.cpp.o
[ 38%] Building CUDA object CMakeFiles/encode.dir/encode.cpp.o
[ 41%] Linking CUDA executable encode
[ 66%] Built target encode
[ 69%] Building CUDA object CMakeFiles/verify.dir/source/dataset.cpp.o
[ 72%] Building CUDA object CMakeFiles/verify.dir/verify.cpp.o
[ 75%] Linking CUDA executable verify
[100%] Built target verify


In [ ]:
!./Build/encode gpu models/model.dat cifar-10-batches-bin

[Config] Failed to open configuration file: config.yaml 
[Config] Using default configuration values 
[Config] Current Configuration: 
[Config] Batch size: 64 
[Config] Epochs: 20 
[Config] Learning rate: 0.001 
[Config] Seed: 0 
[Config] Checkpoint interval: 50 
[Config] Tensor block size: 16x16 
[Config] Conv2D block size: 16x16 
[Config] MaxPool2D block size: 16x16 
[Config] Upsample2D block size: 16x16 
[Main] Using GPU for encoding 
[Autoencoder<Device::GPU>] Model built with 9 layers 
[Autoencoder<Device::GPU>] Model loaded from models/model.dat 
[Dataset] Initializing Dataset... 
[Dataset] Loading training data... 
[Dataset] Loading test data... 
[Dataset] Dataset ready. Train: 50000 Test: 10000 
[Main] Starting feature extraction on training set with  50000 samples 
[Main] Processed batch 10 / 196 | Batch time: 109.922 ms 
[Main] Processed batch 20 / 196 | Batch time: 112.333 ms 
[Main] Processed batch 30 / 196 | Batch time: 115.074 ms 
[Main] Processed batch 40 / 196 | Batch t

In [ ]:
import numpy as np
import cupy as cp

from cuml.svm import SVC
from cuml.preprocessing import StandardScaler
from cuml.metrics import accuracy_score

# =====================================================
# 1. LOAD FEATURES + LABELS
# =====================================================

# Train set
X_train = np.fromfile(
    "/content/content/output/train_features.bin",
    dtype=np.float32
).reshape(50000, 8192)

y_train = np.fromfile(
    "/content/content/output/train_labels.bin",
    dtype=np.uint16
)

# Test set
X_test = np.fromfile(
    "/content/content/output/test_features.bin",
    dtype=np.float32
).reshape(10000, 8192)

y_test = np.fromfile(
    "/content/content/output/test_labels.bin",
    dtype=np.uint16
)

assert X_train.shape == (50000, 8192)
assert X_test.shape  == (10000, 8192)
assert y_train.shape == (50000,)
assert y_test.shape  == (10000,)

print("Train X:", X_train.shape, "Train y:", y_train.shape)
print("Test  X:", X_test.shape,  "Test  y:", y_test.shape)

# =====================================================
# 2. MOVE TO GPU
# =====================================================

X_train = cp.asarray(X_train)
X_test  = cp.asarray(X_test)
y_train = cp.asarray(y_train)
y_test  = cp.asarray(y_test)

# =====================================================
# 3. STANDARD SCALER
# =====================================================

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# =====================================================
# 4. TRAIN SVM
# =====================================================

svm = SVC(
    kernel="rbf",
    C=10,
    gamma="auto"
)

svm.fit(X_train, y_train)

# =====================================================
# 5. EVALUATION
# =====================================================

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("SVM Test Accuracy:", float(acc))

Train X: (50000, 8192) Train y: (50000,)
Test  X: (10000, 8192) Test  y: (10000,)
SVM Test Accuracy: 0.6692


In [ ]:
import numpy as np

np.savez(
    "svm_cuml_model.npz",
    support_vectors=svm.support_vectors_,
    dual_coef=svm.dual_coef_,
    intercept=svm.intercept_,
    classes=svm.classes_
)

print("SVM model saved to svm_cuml_model.npz")

SVM model saved to svm_cuml_model.npz
